In [1]:
# ============================================================
# Step 1: Check CUDA and GPU Availability
# ============================================================

# This command shows the NVIDIA GPU assigned to this Colab runtime.
# It also displays the CUDA driver version and GPU memory information.
!nvidia-smi

# This command checks whether the NVIDIA CUDA compiler (nvcc)
# is installed and shows the CUDA toolkit version.
!nvcc --version

Mon Aug 31 19:58:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
# ============================================================
# Student-specific assignment settings
# ============================================================

SID4 = 1980
SEED = 1980
SLICE = 980
HP_ID = 0
CLS_A = 0
CLS_B = 1

print("SID4 =", SID4)
print("SEED =", SEED)
print("SLICE =", SLICE)
print("HP_ID =", HP_ID)
print("CLS_A =", CLS_A)
print("CLS_B =", CLS_B)

SID4 = 1980
SEED = 1980
SLICE = 980
HP_ID = 0
CLS_A = 0
CLS_B = 1


In [2]:
%%writefile matrix_mul.cu

// ============================================================
// Step 2: CUDA Matrix Multiplication Program
// ============================================================

#include <iostream>
#include <cuda_runtime.h>

// CUDA kernel for matrix multiplication.
// Each GPU thread computes one element of matrix C.
__global__ void matrixMulKernel(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    // Calculate the row handled by this thread.
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    // Calculate the column handled by this thread.
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    // Make sure the thread is inside the matrix dimensions.
    if (row < N && col < N) {

        float sum = 0.0f;

        // Compute one element of the output matrix.
        // C[row, col] = sum of A[row, k] * B[k, col]
        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }

        // Store the calculated value in matrix C.
        C[row * N + col] = sum;
    }
}

Writing matrix_mul.cu


In [3]:
%%writefile -a matrix_mul.cu

// ============================================================
// CPU Matrix Multiplication
// ============================================================

// This function performs matrix multiplication on the CPU.
// It is used as the baseline for comparing GPU performance.
void matrixMulCPU(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    // Loop through every row of matrix A.
    for (int row = 0; row < N; row++) {

        // Loop through every column of matrix B.
        for (int col = 0; col < N; col++) {

            float sum = 0.0f;

            // Multiply corresponding elements and accumulate them.
            for (int k = 0; k < N; k++) {
                sum += A[row * N + k] * B[k * N + col];
            }

            // Store the calculated value in output matrix C.
            C[row * N + col] = sum;
        }
    }
}

Appending to matrix_mul.cu


In [4]:
%%writefile -a matrix_mul.cu

// ============================================================
// Main Function
// Allocates matrices and initializes input values
// ============================================================

int main(int argc, char** argv) {

    // Default matrix size.
    int N = 256;

    // If a matrix size is provided from the command line,
    // use that value instead.
    if (argc > 1) {
        N = atoi(argv[1]);
    }

    // Total number of elements in one N x N matrix.
    size_t numElements = (size_t)N * N;

    // Total memory required for one matrix in bytes.
    size_t bytes = numElements * sizeof(float);

    // Allocate CPU memory for input matrices A and B,
    // CPU result matrix, and GPU result matrix.
    float* h_A = new float[numElements];
    float* h_B = new float[numElements];
    float* h_C_cpu = new float[numElements];
    float* h_C_gpu = new float[numElements];

    // Initialize input matrices with simple values.
    // Using fixed values makes the computation reproducible.
    for (size_t i = 0; i < numElements; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 1.0f;
    }

    // Print the matrix size being tested.
    std::cout << "Matrix size: "
              << N << " x " << N << std::endl;

Appending to matrix_mul.cu


In [5]:
%%writefile -a matrix_mul.cu

// ============================================================
// Step 5: Allocate GPU Memory and Copy Inputs to GPU
// ============================================================

// Device pointers for matrices A, B, and C.
float* d_A;
float* d_B;
float* d_C;

// Allocate memory on the GPU for all three matrices.
cudaMalloc((void**)&d_A, bytes);
cudaMalloc((void**)&d_B, bytes);
cudaMalloc((void**)&d_C, bytes);

// Copy input matrix A from CPU memory to GPU memory.
cudaMemcpy(
    d_A,
    h_A,
    bytes,
    cudaMemcpyHostToDevice
);

// Copy input matrix B from CPU memory to GPU memory.
cudaMemcpy(
    d_B,
    h_B,
    bytes,
    cudaMemcpyHostToDevice
);

Appending to matrix_mul.cu


In [6]:
%%writefile -a matrix_mul.cu

// ============================================================
// Step 6: Configure CUDA Blocks and Threads
// and Launch the GPU Kernel
// ============================================================

// Each CUDA block contains 16 x 16 = 256 threads.
// Each thread computes one element of output matrix C.
dim3 threadsPerBlock(16, 16);

// Calculate how many blocks are required to cover
// the complete N x N output matrix.
dim3 blocksPerGrid(
    (N + threadsPerBlock.x - 1) / threadsPerBlock.x,
    (N + threadsPerBlock.y - 1) / threadsPerBlock.y
);

// Launch the CUDA matrix multiplication kernel.
//
// <<<blocksPerGrid, threadsPerBlock>>>
// tells CUDA how many blocks and threads to use.
matrixMulKernel<<<blocksPerGrid, threadsPerBlock>>>(
    d_A,
    d_B,
    d_C,
    N
);

// Wait until all GPU threads have finished.
// This is important before reading the GPU result.
cudaDeviceSynchronize();

// Copy the completed result matrix C
// from GPU memory back to CPU memory.
cudaMemcpy(
    h_C_gpu,
    d_C,
    bytes,
    cudaMemcpyDeviceToHost
);

Appending to matrix_mul.cu


In [7]:
%%writefile -a matrix_mul.cu

// ============================================================
// Step 7: Run CPU Version and Check Correctness
// ============================================================

// Run the same matrix multiplication on the CPU.
matrixMulCPU(
    h_A,
    h_B,
    h_C_cpu,
    N
);

// Compare the CPU and GPU results.
// If both implementations are correct, corresponding
// elements should be approximately equal.
bool correct = true;

for (size_t i = 0; i < numElements; i++) {

    // Small tolerance is used because floating-point
    // calculations can have tiny numerical differences.
    if (fabs(h_C_cpu[i] - h_C_gpu[i]) > 1e-3) {
        correct = false;
        break;
    }
}

// Print whether the GPU computation matches the CPU result.
if (correct) {
    std::cout << "Result check: PASSED" << std::endl;
} else {
    std::cout << "Result check: FAILED" << std::endl;
}

Appending to matrix_mul.cu


In [8]:
%%writefile -a matrix_mul.cu

// ============================================================
// Step 8: Timing Measurements
// ============================================================

// ---------- CPU timing ----------

// CUDA events are not used for CPU timing.
// We use the C++ high-resolution clock instead.
auto cpu_start = std::chrono::high_resolution_clock::now();

matrixMulCPU(
    h_A,
    h_B,
    h_C_cpu,
    N
);

auto cpu_end = std::chrono::high_resolution_clock::now();

// Convert CPU execution time to milliseconds.
double cpu_ms =
    std::chrono::duration<double, std::milli>(
        cpu_end - cpu_start
    ).count();


// ---------- GPU timing ----------

// Create CUDA events for accurate GPU timing.
cudaEvent_t start, stop;

cudaEventCreate(&start);
cudaEventCreate(&stop);


// Measure Host-to-Device transfer time.
cudaEventRecord(start);

cudaMemcpy(
    d_A,
    h_A,
    bytes,
    cudaMemcpyHostToDevice
);

cudaMemcpy(
    d_B,
    h_B,
    bytes,
    cudaMemcpyHostToDevice
);

cudaEventRecord(stop);
cudaEventSynchronize(stop);

float h2d_ms = 0.0f;
cudaEventElapsedTime(&h2d_ms, start, stop);


// Measure only the GPU kernel execution time.
cudaEventRecord(start);

matrixMulKernel<<<blocksPerGrid, threadsPerBlock>>>(
    d_A,
    d_B,
    d_C,
    N
);

cudaEventRecord(stop);
cudaEventSynchronize(stop);

float kernel_ms = 0.0f;
cudaEventElapsedTime(&kernel_ms, start, stop);


// Measure Device-to-Host transfer time.
cudaEventRecord(start);

cudaMemcpy(
    h_C_gpu,
    d_C,
    bytes,
    cudaMemcpyDeviceToHost
);

cudaEventRecord(stop);
cudaEventSynchronize(stop);

float d2h_ms = 0.0f;
cudaEventElapsedTime(&d2h_ms, start, stop);


// Total GPU transfer time.
float transfer_ms = h2d_ms + d2h_ms;

// End-to-end GPU time includes kernel + transfers.
float gpu_total_ms = kernel_ms + transfer_ms;

// Calculate end-to-end speedup.
double speedup = cpu_ms / gpu_total_ms;


// Print timing results.
std::cout << "CPU time (ms): " << cpu_ms << std::endl;
std::cout << "GPU kernel time (ms): " << kernel_ms << std::endl;
std::cout << "H2D + D2H time (ms): " << transfer_ms << std::endl;
std::cout << "GPU end-to-end time (ms): " << gpu_total_ms << std::endl;
std::cout << "End-to-end speedup: " << speedup << "x" << std::endl;

Appending to matrix_mul.cu


In [9]:
# ============================================================
# Step 9: Add Required C++ Headers
# ============================================================

# Read the current CUDA source file.
with open("matrix_mul.cu", "r") as file:
    code = file.read()

# Add the required headers after #include <iostream>.
# chrono is needed for CPU timing.
# cmath is needed for fabs() during result verification.
code = code.replace(
    "#include <iostream>",
    "#include <iostream>\n#include <chrono>\n#include <cmath>\n#include <cstdlib>"
)

# Save the updated CUDA source file.
with open("matrix_mul.cu", "w") as file:
    file.write(code)

print("Required headers added successfully.")

Required headers added successfully.


In [10]:
%%writefile -a matrix_mul.cu

// ============================================================
// Step 10: Clean Up Memory and Finish Program
// ============================================================

// Destroy CUDA timing events.
cudaEventDestroy(start);
cudaEventDestroy(stop);

// Free memory allocated on the GPU.
cudaFree(d_A);
cudaFree(d_B);
cudaFree(d_C);

// Free memory allocated on the CPU.
delete[] h_A;
delete[] h_B;
delete[] h_C_cpu;
delete[] h_C_gpu;

// Reset the CUDA device and release remaining GPU resources.
cudaDeviceReset();

// Return 0 means the program completed successfully.
return 0;

}  // End of main()

Appending to matrix_mul.cu


In [11]:
# ============================================================
# Step 11: Compile the CUDA C Program
# ============================================================

# nvcc is NVIDIA's CUDA compiler.
# It compiles matrix_mul.cu into an executable named matrix_mul.
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [12]:
# ============================================================
# Step 12: Test Matrix Size 256 x 256
# ============================================================

# Run the compiled CUDA program using N = 256.
# This will execute both CPU and GPU matrix multiplication
# and print the correctness check and timing measurements.
!./matrix_mul 256

Matrix size: 256 x 256
Result check: PASSED
CPU time (ms): 53.1006
GPU kernel time (ms): 0.137664
H2D + D2H time (ms): 0.246784
GPU end-to-end time (ms): 0.384448
End-to-end speedup: 138.122x


In [13]:
# ============================================================
# Step 13: Test Matrix Size 1024 x 1024
# ============================================================

# Run the CUDA program with a 1024 x 1024 matrix.
!./matrix_mul 1024

Matrix size: 1024 x 1024
Result check: PASSED
CPU time (ms): 8214.69
GPU kernel time (ms): 9.20989
H2D + D2H time (ms): 2.95552
GPU end-to-end time (ms): 12.1654
End-to-end speedup: 675.25x


In [14]:
# ============================================================
# Step 14: Test Matrix Size 4096 x 4096
# ============================================================

# Run the CUDA program using a 4096 x 4096 matrix.
# This is the largest matrix size required by the assignment.
#
# The program will report:
# 1. CPU execution time
# 2. GPU kernel execution time
# 3. Host-to-Device + Device-to-Host transfer time
# 4. GPU end-to-end time
# 5. End-to-end GPU speedup
!./matrix_mul 4096

Matrix size: 4096 x 4096
Result check: PASSED
^C


In [15]:
# ============================================================
# Step 15: Replace Slow CPU Matrix Multiplication
# with a Cache-Friendly Version
# ============================================================

# Read the current CUDA source code.
with open("matrix_mul.cu", "r") as file:
    code = file.read()

# Find the old CPU matrix multiplication function.
start = code.index("void matrixMulCPU(")

# Find where that function ends.
end_marker = "\n}\n"
end = code.index(end_marker, start) + len(end_marker)

# New cache-friendly CPU implementation.
# It still performs ordinary matrix multiplication on the CPU,
# but changes the loop order from row-col-k to row-k-col.
#
# This accesses matrix B sequentially in memory,
# which is much faster for large matrices.
new_cpu_function = r'''
void matrixMulCPU(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    // Initialize output matrix C to zero.
    for (size_t i = 0; i < (size_t)N * N; i++) {
        C[i] = 0.0f;
    }

    // Cache-friendly loop order: row -> k -> column.
    for (int row = 0; row < N; row++) {

        for (int k = 0; k < N; k++) {

            // Read this value from A once and reuse it.
            float a_value = A[row * N + k];

            for (int col = 0; col < N; col++) {

                // Accumulate the matrix multiplication result.
                C[row * N + col] +=
                    a_value * B[k * N + col];
            }
        }
    }
}
'''

# Replace the old CPU function with the optimized version.
code = code[:start] + new_cpu_function + code[end:]

# Save the updated CUDA source file.
with open("matrix_mul.cu", "w") as file:
    file.write(code)

print("CPU matrix multiplication updated successfully.")

CPU matrix multiplication updated successfully.


In [16]:
# ============================================================
# Step 16: Recompile Updated CUDA Program
# ============================================================

# Recompile the CUDA source after changing the CPU implementation.
# The output executable will again be named matrix_mul.
!nvcc matrix_mul.cu -o matrix_mul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [17]:
# ============================================================
# Step 17: Re-test Matrix Size 256 x 256
# After CPU optimization
# ============================================================

# Run the updated program for a 256 x 256 matrix.
# These new results will replace the earlier 256 timings
# so all final comparisons use the same implementation.
!./matrix_mul 256

Matrix size: 256 x 256
Result check: PASSED
CPU time (ms): 53.5137
GPU kernel time (ms): 0.133152
H2D + D2H time (ms): 0.247936
GPU end-to-end time (ms): 0.381088
End-to-end speedup: 140.424x


In [18]:
# ============================================================
# Step 18: Re-test Matrix Size 1024 x 1024
# After CPU optimization
# ============================================================

# Run the updated program for a 1024 x 1024 matrix.
# These results will be used in the final comparison table.
!./matrix_mul 1024

Matrix size: 1024 x 1024
Result check: PASSED
CPU time (ms): 3491.75
GPU kernel time (ms): 9.21133
H2D + D2H time (ms): 2.90189
GPU end-to-end time (ms): 12.1132
End-to-end speedup: 288.26x


In [19]:
# ============================================================
# Step 19: Re-test Matrix Size 4096 x 4096
# After CPU optimization
# ============================================================

# Run the updated program for the largest required matrix.
# The cache-friendly CPU implementation should complete
# substantially faster than the previous version.
!./matrix_mul 4096

Matrix size: 4096 x 4096
Result check: PASSED
CPU time (ms): 248490
GPU kernel time (ms): 417.659
H2D + D2H time (ms): 45.8121
GPU end-to-end time (ms): 463.471
End-to-end speedup: 536.15x


In [20]:
# ============================================================
# Step 20: Check Available NVIDIA Profilers
# ============================================================

# Check whether Nsight Systems is installed.
!which nsys

# Check whether Nsight Compute is installed.
!which ncu

# Check whether the older nvprof profiler is installed.
!which nvprof

/usr/local/cuda/bin/ncu
/usr/local/cuda/bin/nvprof


In [21]:
# ============================================================
# Step 21: Profile CUDA Kernel with Nsight Compute
# Matrix Size: 1024 x 1024
# ============================================================

# Run the CUDA program through Nsight Compute.
# This profiles the GPU kernel and reports execution details.
#
# --set full collects a detailed set of kernel metrics.
# We use 1024 x 1024 so profiling finishes faster than 4096.
!ncu --set full ./matrix_mul 1024

Matrix size: 1024 x 1024
==PROF== Connected to process 24467 (/content/matrix_mul)
==PROF== Profiling "matrixMulKernel" - 0: 0%....50%....100% - 31 passes
Result check: PASSED
==PROF== Profiling "matrixMulKernel" - 1: 0%....50%....100% - 31 passes
CPU time (ms): 6504.56
GPU kernel time (ms): 6786.26
H2D + D2H time (ms): 3.04262
GPU end-to-end time (ms): 6789.3
End-to-end speedup: 0.95806x
==PROF== Disconnected from process 24467
[24467] matrix_mul@127.0.0.1
  matrixMulKernel(const float *, const float *, float *, int) (64, 64, 1)x(16, 16, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         5.00
    SM Frequency                    Mhz       585.00
    Elapsed Cycles                cycle    5,381,228
    Memory Throughput                 %        62.45

In [22]:
# ============================================================
# Step 22: Profile Kernel and Memory Transfers
# Matrix Size: 1024 x 1024
# Profiler: nvprof
# ============================================================

# nvprof summarizes CUDA GPU activities.
# It can separately show:
# - matrixMulKernel execution
# - Host-to-Device (HtoD) memory copies
# - Device-to-Host (DtoH) memory copies
#
# We use 1024 x 1024 to keep profiling time reasonable.
!nvprof ./matrix_mul 1024

Matrix size: 1024 x 1024
==24887== NVPROF is profiling process 24887, command: ./matrix_mul 1024
Result check: PASSED
CPU time (ms): 4695.39
GPU kernel time (ms): 9.20592
H2D + D2H time (ms): 3.06938
GPU end-to-end time (ms): 12.2753
End-to-end speedup: 382.507x
==24887== Profiling application: ./matrix_mul 1024
==24887== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   77.72%  18.345ms         2  9.1725ms  9.1723ms  9.1728ms  matrixMulKernel(float const *, float const *, float*, int)
                   13.09%  3.0903ms         4  772.58us  731.35us  799.87us  [CUDA memcpy HtoD]
                    9.19%  2.1695ms         2  1.0847ms  657.59us  1.5119ms  [CUDA memcpy DtoH]
      API calls:   55.62%  175.15ms         3  58.384ms  57.622us  175.03ms  cudaMalloc
                   34.73%  109.36ms         1  109.36ms  109.36ms  109.36ms  cudaDeviceReset
                    2.94%  9.2519ms         3  3.0840ms  2.8140us  

In [23]:
# ============================================================
# Step 23: Final CUDA Performance Results
# ============================================================

import pandas as pd

# Store the timing results from the normal, non-profiler runs.
# These are the results we will report in the assignment.
cuda_results = pd.DataFrame({
    "Matrix Size": [
        "256 x 256",
        "1024 x 1024",
        "4096 x 4096"
    ],

    "CPU Time (ms)": [
        53.5137,
        3491.75,
        248490.0
    ],

    "GPU Kernel Time (ms)": [
        0.133152,
        9.21133,
        417.659
    ],

    "H2D + D2H Time (ms)": [
        0.247936,
        2.90189,
        45.8121
    ],

    "GPU End-to-End Time (ms)": [
        0.381088,
        12.1132,
        463.471
    ],

    "End-to-End Speedup": [
        140.424,
        288.26,
        536.15
    ]
})

# Display the final comparison table.
cuda_results

,Matrix Size,CPU Time (ms),GPU Kernel Time (ms),H2D + D2H Time (ms),GPU End-to-End Time (ms),End-to-End Speedup
0,256 x 256,53.5137,0.133152,0.247936,0.381088,140.424
1,1024 x 1024,3491.7500,9.211330,2.901890,12.113200,288.260
2,4096 x 4096,248490.0000,417.659000,45.812100,463.471000,536.150


Performance Analysis:
The GPU achieved an end-to-end speedup for all three tested matrix sizes, with the smallest beneficial size being 256 × 256, where the speedup was 140.424×. The speedup increased to 288.260× for 1024 × 1024 and 536.150× for 4096 × 4096 as the computational workload increased. The crossover point is not zero because GPU execution introduces fixed overheads such as kernel launching and transferring data between host and device memory. For sufficiently small workloads, these overheads can outweigh the benefit of GPU parallelism, whereas larger matrix multiplications provide enough parallel computation to amortize these costs.

In [25]:
# ============================================================
# Step 26: Verify CUDA source file for submission
# ============================================================

# Check that the CUDA source file exists in the Colab workspace.
import os

print("matrix_mul.cu exists:", os.path.exists("matrix_mul.cu"))
print("File size:", os.path.getsize("matrix_mul.cu"), "bytes")

matrix_mul.cu exists: True
File size: 8769 bytes


In [26]:
# Download the CUDA source file before the runtime disconnects

from google.colab import files

files.download("matrix_mul.cu")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>